In [ ]:
!pip install -q groq chromadb sentence-transformers pypdf rank-bm25

In [ ]:
import os
import re
import hashlib
from pathlib import Path
from groq import Groq
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from pypdf import PdfReader
from rank_bm25 import BM25Okapi

os.environ["GROQ_API_KEY"] = "groq_api"

In [ ]:
def load_and_chunk_pdfs(pdf_paths, chunk_size=500, overlap=50):
    chunks, metadatas, ids = [], [], []

    for path in pdf_paths:
        reader = PdfReader(path)
        fname  = Path(path).name

        for page_num, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            # Split into overlapping chunks
            start = 0
            while start < len(text):
                end     = start + chunk_size
                chunk   = text[start:end].strip()
                if len(chunk) > 50:  # skip tiny chunks
                    uid = hashlib.md5(
                        f"{fname}{page_num}{chunk[:30]}".encode()
                    ).hexdigest()
                    chunks.append(chunk)
                    metadatas.append({
                        "filename"   : fname,
                        "page"       : page_num,
                        "page_label" : f"p.{page_num + 1}",
                        "source"     : str(path),
                    })
                    ids.append(uid)
                start += chunk_size - overlap

    print(f"Loaded {len(chunks)} chunks from {len(pdf_paths)} files")
    return chunks, metadatas, ids

In [ ]:
def build_vector_store(chunks, metadatas, ids, persist_dir="./chroma_db"):
    ef = SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    client     = chromadb.PersistentClient(path=persist_dir)
    collection = client.get_or_create_collection(
        name               = "rag_docs",
        embedding_function = ef,
        metadata           = {"hnsw:space": "cosine"},
    )
    # Add in batches of 100
    for i in range(0, len(chunks), 100):
        collection.upsert(
            documents = chunks[i:i+100],
            metadatas = metadatas[i:i+100],
            ids       = ids[i:i+100],
        )
    print(f"Vector store ready — {collection.count()} vectors")
    return collection, ef

In [ ]:
def hybrid_retrieve(query, collection, chunks, metadatas, k=3):
    # Vector search
    vec_results = collection.query(query_texts=[query], n_results=k*3)
    vec_docs     = vec_results["documents"][0]
    vec_metas    = vec_results["metadatas"][0]

    # BM25 search
    tokenized    = [c.lower().split() for c in chunks]
    bm25         = BM25Okapi(tokenized)
    bm25_scores  = bm25.get_scores(query.lower().split())
    top_bm25_idx = sorted(range(len(bm25_scores)),
                          key=lambda i: bm25_scores[i], reverse=True)[:k*3]

    # RRF merge
    scores  = {}
    doc_map = {}

    for rank, (doc, meta) in enumerate(zip(vec_docs, vec_metas)):
        key = meta["filename"] + str(meta["page"])
        scores[key]  = scores.get(key, 0) + 1/(60 + rank + 1)
        doc_map[key] = (doc, meta)

    for rank, idx in enumerate(top_bm25_idx):
        meta = metadatas[idx]
        key  = meta["filename"] + str(meta["page"])
        scores[key]  = scores.get(key, 0) + 1/(60 + rank + 1)
        doc_map[key] = (chunks[idx], meta)

    sorted_keys = sorted(scores, key=lambda k: scores[k], reverse=True)[:k]
    return [doc_map[k] for k in sorted_keys]

In [ ]:
SYSTEM_PROMPT = """You are a precise document assistant.
Answer ONLY using the context below.
If the answer is not in the context, say: "I couldn't find that in the uploaded documents."
Reference the source naturally in your answer e.g. (source: filename.pdf, p.3).

CONTEXT:
{context}"""

client_groq = Groq(api_key=os.environ["GROQ_API_KEY"])
chat_history = []

def rag_query(question, collection, chunks, metadatas, k=3):
    # Retrieve
    results = hybrid_retrieve(question, collection, chunks, metadatas, k)

    # Build context
    context  = "\n\n---\n\n".join([doc for doc, _ in results])
    citations = [{
        "filename"  : meta["filename"],
        "page_label": meta["page_label"],
        "preview"   : doc[:150],
    } for doc, meta in results]

    # Build messages with history
    messages = [{"role": "system", "content": SYSTEM_PROMPT.format(context=context)}]
    # Add last 3 exchanges from history
    messages += chat_history[-6:]
    messages.append({"role": "user", "content": question})

    # Call Groq
    response = client_groq.chat.completions.create(
        model       = "llama-3.1-8b-instant",
        messages    = messages,
        temperature = 0.2,
        max_tokens  = 1024,
    )
    answer = response.choices[0].message.content

    # Update history
    chat_history.append({"role": "user",      "content": question})
    chat_history.append({"role": "assistant", "content": answer})
    # Keep only last 6 messages (3 exchanges)
    if len(chat_history) > 6:
        chat_history.pop(0)
        chat_history.pop(0)

    return {"answer": answer, "citations": citations}

In [ ]:
from google.colab import files
uploaded = files.upload()

os.makedirs("/content/docs", exist_ok=True)
pdf_paths = []
for fname in uploaded:
    dest = f"/content/docs/{fname}"
    os.rename(fname, dest)
    pdf_paths.append(dest)

chunks, metadatas, ids = load_and_chunk_pdfs(pdf_paths)
collection, ef         = build_vector_store(
    chunks, metadatas, ids,
    persist_dir="/content/drive/MyDrive/rag_chroma_db"
)
print("Pipeline ready!")

In [ ]:
result = rag_query(
    "what is the main topic of these documents?",
    collection, chunks, metadatas
)
print(result["answer"])
print("\nSources:")
for c in result["citations"]:
    print(f"  → {c['filename']} {c['page_label']}")
    print(f"    \"{c['preview']}...\"")